<a href="https://colab.research.google.com/github/snr-laboratory/snrlab-ic-q-pix-v1/blob/chip_network_sim/chip_network_sim/dev_journals/kgosine_202603_architecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Overview


This program simulates an m x n chip network. Each chip can be described using only RTL and chips communicate via a lightweight messaging API. Each chip can recieve data from one upstream neighbor, send data to one downstream neighbor, and receives timing information from a shared launcher. Each chip also generates data locally such that data traffic is buffered using a single local FIFO. A network of chips are launched by the Orchestrator (orchestrator.c) which coordinates timestep ticks for the network and configurations for the array.

nanomsg-next-gen (nng) is a lightweight brokerless API which handles the movement of data packets among chips. Each chip has 4 nng sockets (specified with tcp URLs) which transmit data between neighboring chips, between chips and the orchestrator, or between chips and a metric collector. Importantly, this form of messaging occurs over a network of distributed CPUs (nodes), each simulating a single chip.

Verilator is used during the build process to convert RTL (and in future could be used in a mixed signal cosimualtion discussed later) to a C++ model of the same behavior to which clock edges (ticks) can be passed.

The entire program is a distributed synchronous hardware simulator where there is a global clock, chips which act as hardware modules, and nng sockets which act as wires between hardware modules.


## Chip Unit

A chip functions as a 2-input, 1-output unit. The first input is data from one upstream chip. The second input is data generated locally on the chip. The one output exits the chip's FIFO and is passed to the downstream chip.

The FIFO on the chip is defined with RTL (2-input, 1-output) which buffers data trafficked through the chip and acts as an arbiter between locally generated and pass-through data (data from upstream chip).

The locally generated data can be viewed as the output of any analog or mixed-signal front-end structure within the chip which generates data-packets. In this example, this front-end is generalized to a random number generator (defined in software) which generates data packets. The structure of a data packet is as follows:

* 64-bits long
* 16-bit chip_id
* 24-bit timestamp
* 24-bit data payload (randomly generated string)

The rate at which data is locally generated in a chip is configurable via a gen_ppm argument. On every clock TICK, the chip generates a random number between 0-1,000,000 which is compared with the gen_ppm input. If the randomly generated number is less than the gen_ppm number, a data packet is created during that timestep (otherwise no data packet is generated).  

A chip is defined primarily in the script chip_rtl.cpp which does the following:

* take in the Verilated RTL model for the chip (model.cpp)
* communicate with the upstream chip to receive upstream data packet
* pass upstream data packet as input to the chip model (model.cpp)
* (may) generate a local data packet and pass as an input to the chip model (model.cpp)
* simulate one complete time step of the chip model
* communicate with the downstream chip to pass output data packet
* communicate with Orchestrator to receive timestep TICK and send DONE reply
* send performance metrics to a metric collector





<img src="https://drive.google.com/uc?export=view&id=1F61Gr81PULjLpJPcJ0dODnka5_2mqnqU" width="600">

Communication with other chips is handled via NNG sockets which are initialized in the chip. Details about message handoff below.

## Message Transport

<img src="https://drive.google.com/uc?export=view&id=1dTMjHdmSMudEUrB71ilREtShn7-OCqdJ" height="450">

The program uses NNG (nanomsg-next-gen) as a lightweight brokerless API to handle the movement of data packets between chips. There are four NNG channels defined per chip:

* Timing Channel: Chip <--> Orchestrator

Used for synchronization with the central clock driver provided by the orchestrator; sends each chip a TICK message which cues the chip simulation to run one clock cycle and reply with DONE message when complete.

[ Clock (REQ) --TICK--> Chip (REP) ]

Each chip executes one simulation step

[ Chip (REP) --DONE--> Clock (REQ) ]

Has the added enforcement to WAIT until *all* chips reply DONE to send next tick, ensuring synchronization.

* Communication with Upstream Chip: Upstream_Chip <--> Chip

Upon receiving TICK, the chip will open a REQ socket as request a data packet from the upstream chip. It will wait for a reply from the Upstream Chip. If the Upstream Chip has a data packet, it will send it over the REP socket, otherwise it will reply that is has no data packet to transmit.

[ Chip (REQ) --DATA_REQUEST--> Upstream Chip (REP) ]

[ Upstream Chip (REP) --DATA_REPLY--> Chip (REQ) ]

* Communication with Downstream Chip: Chip <--> Downstream Chip

The chip will wait for a request from the downstream chip and then will respond with DATA_REPLY. If the chip has data to send it will transmit over its REP socket, otherwise it will reply that it has no data to send.  

[ Downstream Chip (REQ) --DATA_REQUEST-->  Chip (REP) ]

[ Chip (REP) --DATA_REPLY--> Downstream Chip (REQ) ]

* PUSH socket: metrics channel which sends one-way reporting to a metrics collector

No blocking or wait for reponse. Used to gather metrics about the simulation.


On each TICK:

* The orchestrator sends TICK to all chips in a network
* The chip will send a request for data to its upstream neighbor and possibly receive data
* The chip will simulate one TICK (one full clock cycle) and publish data to a shared data_server_state_t.
* The chip will wait for a request for data from its downstream neighbor.
* The chip will wait until the main simulation has published its simulation result (which may or may not have generated a packet for output transmission).
* The chip will reply to its downstream neighbor with either an output data packet or that there is no packet to send.



#### Multithreading

A single chip unit can (and will, unless the chip is a sink chip) run two threads. The primary thread does the following:

* sends DATA_REQ to upstream chip (intiated by this chip and serialized with the TICK)
* receives the DATA_REP from upstream chip
* executes RTL timestep simulation
* receives TICK from orchestrator
* sends DONE to orchestrator after simulation step completion
* publish generated data packet to data_server_state_t

The second thread handles downstream communication. This is triggered by the DATA_REQ from the downstream chip which is not necessarily serialized with the tick step and could possibly happen when the primary thread is busy doing its simualtion. This thread is opened for every chip with a downstream neighbor (i.e. only the sink chip will not run this thread).

The two threads communicate via data_server_state_t which is shared state object between them. The primary thread publishes data packets to this shared data server state. The downstream communication thread will wait to reply to the downstream chip until the main thread has published (either a data packet or that there is no data packet) to the data_server_state_t.

pthread is used to handle the synchronization between these two threads on the same chip. This type of multithreading is done on the same process and uses a shared memory space. This parallelization is in contrast to the way multiple chips are simulated in parallel which is done on a truly distributed computing model. The nng socket communication allows distributed computing across CPUs without shared memory.

## Running the Program

The Orchestrator is the top-level simulation driver which parses run settings and launches one chip process per grid cell. It is responsible for running the lock-step control protocol wherein it passes a TICK to all chips and waits to collect DONE from all chips, enforcing that all chips are stepping together in time.

run_from_config.py is an Orchestrator launcher which can pass arguments to the Orchestrator from a configuration file (.json). These confirguration files can define array size, routing, local packet generator rates (per-chip), and fifo depth.

## Applications

* because of the distributed network design of the architecture, chips on the order of those needed in DUNE: for 4x4mm pixels, with 62,500 pixels per m^2. Assuming 64 channel chips, on the order of 10^3 chips/ m^2. Assuming pixelating just one face of a DUNE module, that's 380 m^2 so somewhere around 380,000 chips. the only way to do a simualtion of this scale is with distributed networking.

* taking a design from RTL/Spice (transistor-level) to a tile-level system reduces the number of edge cases which are missed. No part of the simualtion path from chip to tile is simplified, increasing confidence of final product. any unintended effects from the transistor-level or communication design wil appear in the simulation results.

* modifiable to any generic chip: highlighted with this toy model (and later by incorporating analog components), this architecture allows for plugging in any generic chip design or chip routing scheme. Further, the communication system can be tweaked for ACK/NACK signals to be included allowing for testing of new networking schemes with minimal changes to the software. The chip design needs only to be compiled with Verilator once and then can quickly be used in a tile simualtion.

* simple simualtions can show packet loss as a function of FIFO sizes, maximum data generation rates, routing schemes ect.



## Logic Elements

(1) Backpressure via the global clock barrier

Because a chip cannot send DONE until it has finished its simulation tick, anything that gets held up during the TICK pushes back on the whole system. One slow chip slows the entire simulation.

(2) Backpressure on the pull-based data transfer

The downstream chip clint requests data for a specific tick. The simulation is stalled until the upstream chip as simulated and published data for that sequence. If the upstream chip is running behind, the server thread is blocked.

(3) Dropping implemented in the FIFO. If the FIFO is full, incoming packets are dropped.

(4) Backpressure on pass-through packet generation. Locally generated packets take priority for FIFO entry. Therefore, pass through packets will 'hold' and wait to enter the FIFO after the local packet has entered (reducing availability by 1). If both are trying to get into the FIFO at the same time and there is 2 or more available positions in the FIFO, both will enter on the SAME tick. Therefore, arrival of both local and pass through packets at the same time will not cause a data loss unless the FIFO has 0 or 1 position available.

## Test Cases


###(1) Lockstep barrier

The transport is pull-based but storage is single-slot. Correctness relies on seq matching + lockstep. If lockstep breaks, there is no buffering to recover old packets. eg if Chip B (slow) is still working on tick N (hasn't sent DONE) and chip A has published for tick N but then B requests for a tick that doesn't match what Chip A has retained. This would indicate a failure in the lockstep mechanism.

Possible test: artificially slow the downstream so it requests "old" seq. If the clocking barrier is strict this should just slow down the global simualtion.



### Determinism

This program is entirely deterministic (given the same seed for the random number generators). The results of running the same simulation of a 3x5 array of chips 15 times shows that the measured behavioral metrics (including number of transmitted packets, transmission timestamps, packet data, packet loss) are identical across all simulations. There is variation in runtime of approx. 2.7% between runs.

#### Determinism Test Report (3x5 Snake BottomRight->TopLeft)

- Config: `/home/lxusers/k/kalindigosine/snrlab-ic-q-pix-v1/chip_network_sim/config/network_3x5_determinism_snake_br_to_tl.json`
- Runs: 15
- Deterministic behavior check (`delivered_tx + total_drops + per_chip_drops`): **PASS**
- Full output check (including `cycles_per_sec`): **FAIL**

##### Per-run Results

| Run | Delivered Packets (tx) | Total Drops | Cycles/sec | Per-chip Drops (chip_id:count) | Behavior Match vs Run1 | Full Match vs Run1 |
| --- | ---: | ---: | ---: | --- | --- | --- |
| 1 | 261827 | 0 | 2329.463 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | yes |
| 2 | 261827 | 0 | 2266.484 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 3 | 261827 | 0 | 2304.796 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 4 | 261827 | 0 | 2283.983 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 5 | 261827 | 0 | 2245.862 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 6 | 261827 | 0 | 2291.448 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 7 | 261827 | 0 | 2266.732 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 8 | 261827 | 0 | 2192.259 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 9 | 261827 | 0 | 2233.954 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 10 | 261827 | 0 | 2211.480 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 11 | 261827 | 0 | 2203.634 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 12 | 261827 | 0 | 2131.800 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 13 | 261827 | 0 | 2142.861 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 14 | 261827 | 0 | 2169.355 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |
| 15 | 261827 | 0 | 2166.838 | `0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0` | yes | no |

The fastest/slowest runs were 2329.463/2131.800 cycles/sec. Runtime speed varied by ~2.7%, typical for CPU noise.

### FIFO Arbitration Condition (RTL defined)

The RTL for the chip FIFO defines that if a local packet is generated during the same timestep that a pass-through packet enters from the upstream chip, local packet takes priority in entering the FIFO queue. Both packets may enter the FIFO during a single clock tick. The priority condition is evident when there is only one available position in the FIFO such that the local packet should enter and the pass-through packet is dropped.

The test defines a 2 chip network where the downstream chip is the DUT. The gen_ppm of both chips is defined as 1,000,000 such that a local packet is generated each clock tick. The FIFO depth is set to 2. A measure of the local packets dropped versus pass through packets dropped on Chip_1 is recorded.


#### 1x2 Local-Priority Packet-Loss Test Report

- Config: `/home/lxusers/k/kalindigosine/snrlab-ic-q-pix-v1/chip_network_sim/config/network_1x2_priority_loss_test.json`
- Trace run: `traces/priority_loss_1x2/run_1772650434_2818623`
- Run log: `/home/lxusers/k/kalindigosine/snrlab-ic-q-pix-v1/chip_network_sim/reports/priority_loss_1x2/latest_run.log`

##### Test Setup

- Topology: chip `0 -> 1`, with chip `1` as downstream sink (`out_id=-1`)
- Grid: `1x2` chips
- `gen_ppm`: `1,000,000` for both chips (generate every tick)
- `fifo_depth`: `2`
- `ticks`: `2000`

##### Run-Level Metrics (orchestrator)

- delivered packets (`tx`): 1999
- received packets (`rx`): 1999
- locally generated packets (`local`): 4000
- total drops (`drops`): 1998
- fifo peak occupancy (`fifo_peak`): 2
- cycles/sec: 5220.575

##### Packet Loss by Chip (from trace events)

| Chip | Local Enq OK | Neighbor Enq OK | Local Drops | Neighbor Drops | Total Drops | DEQ_OUT |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| 0 | 2000 | 0 | 0 | 0 | 0 | 1999 |
| 1 | 2000 | 1 | 0 | 1998 | 1998 | 1999 |

##### Sink Output Provenance (chip 1, DEQ_OUT events)

- Total sink output packets: 1999
- From chip 1 local traffic (`src_id=1`): 1998
- From chip 0 pass-through traffic (`src_id=0`): 1
- `src_id=0` packet `timestamp` values: `0`
- From other sources: 0

##### Finding

- Interpretation: The only time a pass-through packet is able to enter the FIFO of Chip_1 is at timestep 0 when there are two available possitions in the Chip_1 FIFO. For every following tick, there is only one available position in the FIFO which accepts the locally generated packet over the Chip_0 output packet.

### FIFO Queuing

This test tracks the FIFO queue for a single chip in a run. In this example, the 2x2 network was used where Chip 3 sends traffic into Chip 2 which is generating traffic at a rate sufficient to fill the FIFO and begin dropping packets. The FIFO is 5-wide and the queue is shown below.





| tick | slot_0 | slot_1 | slot_2 | slot_3 | slot_4 |
| --- | --- | --- | --- | --- | --- |
| 0 | 0x0002000000CFCF6B | - | - | - | - |
| 1 | 0x0002000001825438 | 0x000300000044B65A | - | - | - |
| 2 | 0x000300000044B65A | 0x00020000023248A7 | 0x00030000015FA625 | - | - |
| 3 | 0x00020000023248A7 | 0x00030000015FA625 | 0x00020000033C96B2 | 0x0003000002315EFF | - |
| 4 | 0x00030000015FA625 | 0x00020000033C96B2 | 0x0003000002315EFF | 0x00020000043CC553 | 0x0003000003A8A8C6 |
| 5 | 0x00020000033C96B2 | 0x0003000002315EFF | 0x00020000043CC553 | 0x0003000003A8A8C6 | 0x00020000058C00CC |
| 6 | 0x0003000002315EFF | 0x00020000043CC553 | 0x0003000003A8A8C6 | 0x00020000058C00CC | 0x0002000006042E9C |
| 7 | 0x00020000043CC553 | 0x0003000003A8A8C6 | 0x00020000058C00CC | 0x0002000006042E9C | 0x0002000007F57D00 |
| 8 | 0x0003000003A8A8C6 | 0x00020000058C00CC | 0x0002000006042E9C | 0x0002000007F57D00 | 0x00020000082A26B5 |
| 9 | 0x00020000058C00CC | 0x0002000006042E9C | 0x0002000007F57D00 | 0x00020000082A26B5 | 0x0002000009EA2C3A |
| 10 | 0x0002000006042E9C | 0x0002000007F57D00 | 0x00020000082A26B5 | 0x0002000009EA2C3A | 0x000200000A9932CD |

### Congestion wave in a network

Uniform into each chip. For a chained network, should see congestion propagate through the network to chips downstream.

## Packet Tracing in 2x2 Network



![](https://drive.google.com/uc?export=view&id=11gUIXXqPOtzLXKPXAkTtgh4XxxuuSGR_)

## Simulation Metrics


## Future Directions


* incorporating a Spice (analog) portion into the simulation

we've previously successfully performed Spice/RTL cosimulation using Verilator before

* allowing data input from any of the four adjacent chips

this will allow for configuration routes beyond daisy chaining which is obviously vulnerable to single point failure. at this stage, the routing would still be defined at the top level.

* incorpoarting existing RTL (LArPix) including more realistic data packet creation and transmission timing.



cycles/sec run time ect

## Backup

<img src="https://drive.google.com/uc?export=view&id=1KbNMrTRM4rrrpJf15rkbnRgxFLDZpExX" width="600">

